# MedicalPlab Stage-B calibration
Select a T4 or larger GPU. Upload the generated `.stage-b.zip` when prompted.
This notebook is a thin wrapper. It never implements verifier logic or changes
frozen labels. The separate environment avoids changing Colab's base packages.
Results and model revision persist on your mounted Drive for resumability.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
assert (3,11) <= sys.version_info[:2] < (3,13), "Use a Python 3.11/3.12 runtime"
uploaded = files.upload()
assert len(uploaded) == 1
bundle = Path(next(iter(uploaded)))
root = Path('/content/medicalplab-stage-b')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for name in archive.namelist():
        assert (root/name).resolve().is_relative_to(root.resolve()), 'Unsafe archive path'
    archive.extractall(root)
manifest = json.loads((root/'stage_b_bundle_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
os.chdir(root)
drive.mount('/content/drive')
output_dir = Path('/content/drive/MyDrive/MedicalPlab-stage-b')
output_dir.mkdir(exist_ok=True)


In [ ]:
env = Path('/content/medicalplab-stage-b-env')
if not (env/'bin/python').exists():
    subprocess.run([sys.executable,'-m','venv',str(env)],check=True)
python = str(env/'bin/python')
subprocess.run([python,'-m','pip','install','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run([python,'-m','pip','install','-r','requirements-benchmark.txt'],check=True)
subprocess.run([python,'-m','pip','check'],check=True)
os.environ['PYTHONPATH'] = str(root/'src')
subprocess.run([python,'-m','unittest','discover','-s','tests','-t','.','-v'],check=True)
subprocess.run([python,'evaluation/run_stage_b_calibration.py','--preflight'],check=True)


In [ ]:
revision_path = output_dir/'model_revision.txt'
if not revision_path.exists():
    revision = subprocess.check_output([python,'-c',"from huggingface_hub import model_info; print(model_info('Qwen/Qwen3-8B-AWQ').sha)"],text=True).strip()
    revision_path.write_text(revision)
revision = revision_path.read_text().strip()
output = output_dir/'calibration.json'
command = [python,'evaluation/run_stage_b_calibration.py','--revision',revision,'--output',str(output)]
if output.exists(): command += ['--resume']
subprocess.run(command,check=True)
